In [10]:
import os
import glob
import gc
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

In [11]:
# =========================================================
# CONFIG
# =========================================================
DATA_DIR = "archive"
BATCH_SIZE = 256
NUM_EPOCHS = 3
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
TRAIN_VAL_SPLIT = 0.9
NUM_WORKERS = 0
SEED = 42
CHECKPOINT_DIR = "checkpoints_chunked"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


Using device: cuda


In [12]:
# =========================================================
# DATASET FOR ONE NPZ FILE
# =========================================================
class SingleNPZDataset(Dataset):
    def __init__(self, file_path):
        data = np.load(file_path)

        self.features = data["features"].astype(np.float32)
        self.policy_labels = data["policy_labels"].astype(np.int64)
        self.value_labels = data["value_labels"].astype(np.float32)

        assert self.features.shape[1:] == (12, 8, 8)
        assert len(self.features) == len(self.policy_labels) == len(self.value_labels)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        x = torch.tensor(self.features[idx], dtype=torch.float32)
        p = torch.tensor(self.policy_labels[idx], dtype=torch.long)
        v = torch.tensor(self.value_labels[idx], dtype=torch.float32)
        return x, p, v

In [13]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = torch.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = out + residual
        out = torch.relu(out)
        return out


class AlphaZeroChessNet(nn.Module):
    def __init__(self, input_channels=12, num_blocks=12, channels=128, policy_size=4864):
        super().__init__()

        self.input_conv = nn.Conv2d(input_channels, channels, kernel_size=3, padding=1, bias=False)
        self.input_bn = nn.BatchNorm2d(channels)

        self.res_tower = nn.Sequential(*[ResidualBlock(channels) for _ in range(num_blocks)])

        self.policy_conv = nn.Conv2d(channels, 32, kernel_size=1, bias=False)
        self.policy_bn = nn.BatchNorm2d(32)
        self.policy_fc = nn.Linear(32 * 8 * 8, policy_size)

        self.value_conv = nn.Conv2d(channels, 3, kernel_size=1, bias=False)
        self.value_bn = nn.BatchNorm2d(3)
        self.value_fc1 = nn.Linear(3 * 8 * 8, 256)
        self.value_fc2 = nn.Linear(256, 1)

    def forward(self, x):
        x = self.input_conv(x)
        x = self.input_bn(x)
        x = torch.relu(x)

        x = self.res_tower(x)

        p = self.policy_conv(x)
        p = self.policy_bn(p)
        p = torch.relu(p)
        p = p.view(p.size(0), -1)
        policy_logits = self.policy_fc(p)

        v = self.value_conv(x)
        v = self.value_bn(v)
        v = torch.relu(v)
        v = v.view(v.size(0), -1)
        v = torch.relu(self.value_fc1(v))
        value = torch.tanh(self.value_fc2(v)).squeeze(1)

        return policy_logits, value

In [14]:
# =========================================================
# METRICS
# =========================================================
def policy_accuracy(policy_logits, policy_targets):
    preds = torch.argmax(policy_logits, dim=1)
    correct = (preds == policy_targets).sum().item()
    total = policy_targets.size(0)
    return correct / total

In [15]:
# =========================================================
# TRAIN / EVAL HELPERS
# =========================================================
def policy_accuracy(policy_logits, policy_targets):
    preds = torch.argmax(policy_logits, dim=1)
    return (preds == policy_targets).float().mean().item()


def run_epoch(model, loader, optimizer, policy_criterion, value_criterion, train=True):
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_policy_loss = 0.0
    total_value_loss = 0.0
    total_acc = 0.0
    total_samples = 0

    for x, p_targets, v_targets in loader:
        x = x.to(DEVICE, non_blocking=True)
        p_targets = p_targets.to(DEVICE, non_blocking=True)
        v_targets = v_targets.to(DEVICE, non_blocking=True)

        if train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(train):
            policy_logits, value_preds = model(x)

            loss_policy = policy_criterion(policy_logits, p_targets)
            loss_value = value_criterion(value_preds, v_targets)
            loss = loss_policy + loss_value

            if train:
                loss.backward()
                optimizer.step()

        batch_size = x.size(0)
        acc = policy_accuracy(policy_logits, p_targets)

        total_loss += loss.item() * batch_size
        total_policy_loss += loss_policy.item() * batch_size
        total_value_loss += loss_value.item() * batch_size
        total_acc += acc * batch_size
        total_samples += batch_size

    return (
        total_loss / total_samples,
        total_policy_loss / total_samples,
        total_value_loss / total_samples,
        total_acc / total_samples,
    )


In [ ]:
# =========================================================
# MAIN CHUNKED TRAINING LOOP
# =========================================================
def main():
    npz_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.npz")))[:5] # 5 for nowe
    if not npz_files:
        raise FileNotFoundError(f"No .npz files found in {DATA_DIR}")

    model = AlphaZeroChessNet(
        input_channels=12,
        num_blocks=12,
        channels=128,
        policy_size=4864
    ).to(DEVICE)

    policy_criterion = nn.CrossEntropyLoss()
    value_criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    best_global_val_loss = float("inf")

    for epoch in range(1, NUM_EPOCHS + 1):
        print(f"\n{'='*70}")
        print(f"Epoch {epoch}/{NUM_EPOCHS}")
        print(f"{'='*70}")

        random.shuffle(npz_files)

        epoch_train_loss = 0.0
        epoch_train_pol = 0.0
        epoch_train_val = 0.0
        epoch_train_acc = 0.0

        epoch_val_loss = 0.0
        epoch_val_pol = 0.0
        epoch_val_val = 0.0
        epoch_val_acc = 0.0

        num_files_processed = 0

        for file_idx, file_path in enumerate(npz_files, start=1):
            print(f"\n[{file_idx}/{len(npz_files)}] Loading {os.path.basename(file_path)}")

            dataset = SingleNPZDataset(file_path)

            train_size = int(TRAIN_VAL_SPLIT * len(dataset))
            val_size = len(dataset) - train_size

            train_dataset, val_dataset = random_split(
                dataset,
                [train_size, val_size],
                generator=torch.Generator().manual_seed(SEED)
            )

            train_loader = DataLoader(
                train_dataset,
                batch_size=BATCH_SIZE,
                shuffle=True,
                num_workers=NUM_WORKERS,
                pin_memory=(DEVICE.type == "cuda")
            )

            val_loader = DataLoader(
                val_dataset,
                batch_size=BATCH_SIZE,
                shuffle=False,
                num_workers=NUM_WORKERS,
                pin_memory=(DEVICE.type == "cuda")
            )

            train_loss, train_pol_loss, train_val_loss, train_acc = run_epoch(
                model, train_loader, optimizer, policy_criterion, value_criterion, train=True
            )

            val_loss, val_pol_loss, val_val_loss, val_acc = run_epoch(
                model, val_loader, optimizer, policy_criterion, value_criterion, train=False
            )

            print(f"  Train Loss    : {train_loss:.4f}")
            print(f"    Policy Loss : {train_pol_loss:.4f}")
            print(f"    Value Loss  : {train_val_loss:.4f}")
            print(f"    Policy Acc  : {train_acc:.4f}")

            print(f"  Val Loss      : {val_loss:.4f}")
            print(f"    Policy Loss : {val_pol_loss:.4f}")
            print(f"    Value Loss  : {val_val_loss:.4f}")
            print(f"    Policy Acc  : {val_acc:.4f}")

            epoch_train_loss += train_loss
            epoch_train_pol += train_pol_loss
            epoch_train_val += train_val_loss
            epoch_train_acc += train_acc

            epoch_val_loss += val_loss
            epoch_val_pol += val_pol_loss
            epoch_val_val += val_val_loss
            epoch_val_acc += val_acc

            num_files_processed += 1

            # Save rolling checkpoint after each chunk
            rolling_ckpt = os.path.join(CHECKPOINT_DIR, "last_model.pth")
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
            }, rolling_ckpt)

            # Save best checkpoint
            if val_loss < best_global_val_loss:
                best_global_val_loss = val_loss
                best_ckpt = os.path.join(CHECKPOINT_DIR, "best_model.pth")
                torch.save({
                    "epoch": epoch,
                    "file_idx": file_idx,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_loss": val_loss,
                }, best_ckpt)
                print("  Saved new best model.")

            # Explicit cleanup
            del dataset, train_dataset, val_dataset, train_loader, val_loader
            gc.collect()
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

        # Average metrics over files
        epoch_train_loss /= num_files_processed
        epoch_train_pol /= num_files_processed
        epoch_train_val /= num_files_processed
        epoch_train_acc /= num_files_processed

        epoch_val_loss /= num_files_processed
        epoch_val_pol /= num_files_processed
        epoch_val_val /= num_files_processed
        epoch_val_acc /= num_files_processed

        print(f"\nEpoch {epoch} Summary:")
        print(f"  Avg Train Loss    : {epoch_train_loss:.4f}")
        print(f"  Avg Train PolLoss : {epoch_train_pol:.4f}")
        print(f"  Avg Train ValLoss : {epoch_train_val:.4f}")
        print(f"  Avg Train Acc     : {epoch_train_acc:.4f}")

        print(f"  Avg Val Loss      : {epoch_val_loss:.4f}")
        print(f"  Avg Val PolLoss   : {epoch_val_pol:.4f}")
        print(f"  Avg Val ValLoss   : {epoch_val_val:.4f}")
        print(f"  Avg Val Acc       : {epoch_val_acc:.4f}")

    print("\nTraining finished.")


if __name__ == "__main__":
    main()


Epoch 1/3

[1/5] Loading shard_11.npz
  Train Loss    : 3.0362
    Policy Loss : 2.8927
    Value Loss  : 0.1435
    Policy Acc  : 0.2839
  Val Loss      : 2.7831
    Policy Loss : 2.4455
    Value Loss  : 0.3376
    Policy Acc  : 0.3286
  Saved new best model.

[2/5] Loading shard_1.npz
  Train Loss    : 2.4344
    Policy Loss : 2.3242
    Value Loss  : 0.1102
    Policy Acc  : 0.3548
  Val Loss      : 2.4440
    Policy Loss : 2.2653
    Value Loss  : 0.1787
    Policy Acc  : 0.3623
  Saved new best model.

[3/5] Loading shard_10.npz
  Train Loss    : 2.2958
    Policy Loss : 2.1943
    Value Loss  : 0.1015
    Policy Acc  : 0.3779
  Val Loss      : 2.2562
    Policy Loss : 2.1607
    Value Loss  : 0.0956
    Policy Acc  : 0.3845
  Saved new best model.

[4/5] Loading shard_12.npz
  Train Loss    : 2.2195
    Policy Loss : 2.1229
    Value Loss  : 0.0967
    Policy Acc  : 0.3913
  Val Loss      : 2.1778
    Policy Loss : 2.0883
    Value Loss  : 0.0895
    Policy Acc  : 0.3993
  Save

KeyboardInterrupt: 